In [ ]:
import sys
import os
from datetime import datetime
import logging
import pandas as pd
import numpy as np

from typing import Dict, Any, List
import plotly.express as px

from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.signal import find_peaks
from scipy.stats import skew, kurtosis
from sklearn.mixture import GaussianMixture

from core.data_sources import CLOBDataSource
from core.data_structures.candles import Candles

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

logging.getLogger("asyncio").setLevel(logging.CRITICAL)
logging.getLogger("pandas").setLevel(logging.CRITICAL)

In [ ]:
connector_name = "binance"
trading_pair = "USDT-BRL"
interval = "1m"
start_time = datetime(2025, 5, 17).timestamp()
end_time = datetime(2025, 6, 3).timestamp()

fetch = False
clob = CLOBDataSource()
if fetch:
    candles = await clob.get_candles(connector_name, trading_pair, interval, start_time, end_time)
    clob.dump_candles_cache(root_path)
else:
    clob.load_candles_cache(root_path)
    candles = clob.get_candles_from_cache(connector_name, trading_pair, interval)

### Volume target

In [ ]:
connector_name = "binance"
interval = "1d"
start_time = datetime(2024, 1, 1).timestamp()
end_time = datetime(2025, 6, 3).timestamp()
trading_pairs = ["USDT-BRL", "BTC-BRL", "ETH-BRL", "SOL-BRL"]
targets = {
    "1%": 0.01,
    "0.5%": 0.005,
}
window_label = "7d"
window_size = 7

for trading_pair in trading_pairs:
    clob = CLOBDataSource()
    candles = await clob.get_candles(connector_name, trading_pair, interval, start_time, end_time)

    df = candles.data.copy()
    df[f"volume_ma_{window_label}"] = df["quote_asset_volume"].rolling(window=window_size).mean()
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=df.index,
                   y=df.quote_asset_volume,
                   name="Total BRL Volume")
    )
    fig.add_trace(
        go.Scatter(x=df.index,
                   y=df[f"volume_ma_{window_label}"],
                   line_color="yellow",
                   name=f"MA{window_label} BRL Volume")
    )
    for target_label, target_value in targets.items():
        fig.add_trace(
            go.Bar(x=df.index,
                   y=df[f"volume_ma_{window_label}"] * target_value,
                   name=f"{target_label} Target"
            )
        )
    fig.update_yaxes(type="log")
    fig.update_layout(title=f"{trading_pair} Volume 2024 -> Now",
                      height=800, width=1200)
    fig.write_image(f"SD03 - {trading_pair} Volume 2024-now.png")


In [ ]:
df = pd.DataFrame()
for target_label, target_value in targets.items():
    for trading_pair in trading_pairs:
        clob = CLOBDataSource()
        candles = await clob.get_candles(connector_name, trading_pair, interval, start_time, end_time)
        candles.data[f"volume_ma_{window_label}"] = candles.data["quote_asset_volume"].rolling(window=window_size).mean()
        candles.data[f"volume_ma_{window_label}_{target_label}"] = candles.data[f"volume_ma_{window_label}"] * target_value
        candles.data["trading_pair"] = trading_pair
        df = pd.concat([df, candles.data])

    fig = px.box(
        df,
        x="trading_pair",
        y=f"volume_ma_{window_label}_{target_label}",
        points="all",  # optional: show individual points
        title=f"{window_label} Moving Average Volume by Trading Pair ({target_label} Target)"
    )
    fig.update_yaxes(title=f"{window_label} MA Volume (log scale)")
    fig.update_xaxes(title="Trading Pair")
    fig.update_layout(height=800, width=1200)
    fig.write_image(f"SD03 - Comparative targets {target_label}.png")

### Main concept fig declaration

In [ ]:


def main_concept_fig(candles: Candles):
    # Bin prices into ranges to aggregate volume
    price_bins = np.linspace(candles.data['low'].min(), candles.data['high'].max(), 100)
    volume_by_price = pd.cut(candles.data['close'], bins=price_bins)
    volume_agg = candles.data.groupby(volume_by_price)['volume'].sum()

    # Mid-points of bins for y-axis
    price_levels = [interval.mid for interval in volume_agg.index]
    volumes = volume_agg.values

    # Volume-weighted statistics
    total_vol = volumes.sum()
    vwap = np.average(price_levels, weights=volumes)
    std = np.sqrt(np.average((price_levels - vwap)**2, weights=volumes))
    mode_price = price_levels[np.argmax(volumes)]

    zones = {
        "VWAP": vwap,
        "VWAP - STD": vwap - std,
        "VWAP + STD": vwap + std,
        "VWAP - 2STD": vwap - 2*std,
        "VWAP + 2STD": vwap + 2*std,
        "MODE": mode_price
    }


    # Create subplots
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                        column_widths=[0.7, 0.3],
                        horizontal_spacing=0.02)

    # Candlestick chart
    fig.add_trace(go.Scatter(
        x=candles.data.index,
        y=candles.data['close'],
        name="Price"),
        row=1, col=1
    )

    # Horizontal bar chart (volume by price)
    fig.add_trace(go.Bar(
        x=volumes,
        y=price_levels,
        orientation='h',
        name="Volume",
        marker=dict(color='rgba(100, 100, 200, 0.6)')),
        row=1, col=2
    )

    colors = {
        "VWAP": "orange",
        "VWAP - STD": "green",
        "VWAP + STD": "green",
        "VWAP - 2STD": "red",
        "VWAP + 2STD": "red",
        "MODE": "blue"
    }

    for label, y in zones.items():
        fig.add_shape(type="line", x0=candles.data.index[0], x1=candles.data.index[-1], xref='paper',
                      y0=y, y1=y, line=dict(color=colors[label], dash="dash"), row=1, col=1)
        fig.add_annotation(xref="paper", x=candles.data.index[3], y=y, text=label,
                           showarrow=False, yanchor="bottom", font=dict(color=colors[label]), row=1, col=1)


    # Layout tweaks
    fig.update_layout(
        yaxis=dict(title='Price'),
        xaxis=dict(title='Time'),
        xaxis2=dict(title='Volume'),
        height=800,
        showlegend=False
    )

    fig.update_yaxes(row=1, col=2)  # Price axis from top to bottom
    fig.write_image(f"SD03 - Main concept idea.png")
    return fig

main_concept_fig(candles)

### Plot volume profile animation with price over time

In [ ]:

def plot_volume_profile_animation(candles_df: pd.DataFrame, window_size: int, window_step: int = 60 * 24, n_bins: int = 10):
    vol_profile_animation = make_subplots(
        rows=1, cols=2, shared_yaxes=True,
        column_widths=[0.7, 0.3],
        horizontal_spacing=0.02
    )

    colors = {
        "VWAP": "orange",
        "VWAP - STD": "green",
        "VWAP + STD": "green",
        "VWAP - 2STD": "red",
        "VWAP + 2STD": "red",
        "MODE": "blue"
    }

    frames = []

    for i in range(0, len(candles) - window_size, window_step):
        window = candles_df.iloc[i:i+window_size]
        price_bins = np.linspace(window['low'].min(), window['high'].max(), n_bins)
        volume_by_price = pd.cut(window['close'], bins=price_bins)
        volume_agg = window.groupby(volume_by_price)['volume'].sum()

        price_levels = [interval.mid for interval in volume_agg.index]
        volumes = volume_agg.values

        if len(volumes) == 0:
            continue

        vwap = np.average(price_levels, weights=volumes)
        std = np.sqrt(np.average((price_levels - vwap) ** 2, weights=volumes))
        mode_price = price_levels[np.argmax(volumes)]

        zones = {
            "VWAP": vwap,
            "VWAP - STD": vwap - std,
            "VWAP + STD": vwap + std,
            "VWAP - 2STD": vwap - 2 * std,
            "VWAP + 2STD": vwap + 2 * std,
            "MODE": mode_price
        }

        time = window.index
        close = window['close']

        scatter_price = go.Scatter(
            x=time,
            y=close,
            mode='markers',
            marker=dict(size=1),
            # line=dict(color='yellow', shape="hv"),
            name='Close',
            showlegend=False
        )

        bar_volume = go.Bar(
            x=volumes,
            y=price_levels,
            orientation='h',
            marker=dict(color='rgba(100, 100, 200, 0.6)'),
            name='Volume',
            showlegend=False
        )

        shapes = []
        annotations = []

        for label, y in zones.items():
            shapes.append(dict(
                type="line",
                xref="x",
                yref="y",
                x0=time[0],
                x1=time[-1],
                y0=y,
                y1=y,
                line=dict(color=colors[label], dash="dash")
            ))
            annotations.append(dict(
                x=time[0],
                y=y,
                xref='x',
                yref='y',
                text=label,
                showarrow=False,
                yanchor="bottom",
                font=dict(color=colors[label], size=10)
            ))

        frame = go.Frame(
            data=[scatter_price, bar_volume],
            name=str(i),
            layout=go.Layout(shapes=shapes, annotations=annotations)
        )
        frames.append(frame)

    # Initial trace from first frame
    first_frame = frames[0]
    vol_profile_animation.add_trace(first_frame.data[0], row=1, col=1)
    vol_profile_animation.add_trace(first_frame.data[1], row=1, col=2)
    vol_profile_animation.frames = frames

    vol_profile_animation.update_layout(
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[dict(
                label="Play",
                method="animate",
                args=[None, {"frame": {"duration": 0.5, "redraw": False},
                             "fromcurrent": True, "transition": {"duration": 0}}]
            )]
        )],
        yaxis=dict(title='Price'),
        xaxis=dict(title='Time'),
        xaxis2=dict(title='Volume'),
        height=800,
        showlegend=False
    )

    vol_profile_animation.update_yaxes(autorange='reversed', row=1, col=2)

    return vol_profile_animation

window_days = 14
fig = plot_volume_profile_animation(candles.data, 60 * 24 * window_days, window_step=60 * 4)
fig.write_html(
    "volume_profile_light.html",
    include_plotlyjs="cdn",
    full_html=True
)

### Analyze volume distribution over time -> Generate results

In [ ]:
def analyze_window(window: pd.DataFrame, bins: int = 10):
    price_bins = np.linspace(window['low'].min(), window['high'].max(), bins)
    volume_by_price = pd.cut(window['close'], bins=price_bins)
    volume_agg = window.groupby(volume_by_price)['volume'].sum()

    price_levels = [price_bin.mid for price_bin in volume_agg.index]
    volumes = volume_agg.values
    return price_levels, volumes


def analyze_distribution(candles_df):
    results = []
    for i in range(0, len(candles_df) - window_size, window_step):
        window = candles_df.iloc[i:i+window_size]
        price_levels, volumes = analyze_window(window, n_bins)

        if len(volumes) == 0:
            continue

        # VWAP, STD, MODE
        vwap = np.average(price_levels, weights=volumes)
        std = np.sqrt(np.average((price_levels - vwap) ** 2, weights=volumes))
        mode_price = price_levels[np.argmax(volumes)]

        # Skew and Kurtosis
        dist_skewness = skew(volumes)
        dist_kurtosis = kurtosis(volumes)

        # Skew label
        if dist_skewness > 0.5:
            skew_label = "Right-skewed"  # (tail toward high prices)
        elif dist_skewness < -0.5:
            skew_label = "Left-skewed"  # (tail toward low prices)
        else:
            skew_label = "Symmetric"

        # Kurtosis label
        if dist_kurtosis > 0.5:
            kurtosis_label = "Leptokurtic"  # (peaked, fat tails)
        elif dist_kurtosis < -0.5:
            kurtosis_label = "Platykurtic"  # (flat, light tails)
        else:
            kurtosis_label = "Mesokurtic"

        # Peaks
        peaks, _ = find_peaks(volumes, prominence=np.max(volumes) * 0.1)
        num_peaks = len(peaks)
        peaks_prices = [price_levels[i] for i in peaks]
        # GMM Fit
        try:
            gmm = GaussianMixture(n_components=2).fit(volumes.reshape(-1, 1))
            bic = gmm.bic(volumes.reshape(-1, 1))
            gmm_fits = "Bimodal" if bic < 1000 else "Unimodal"
        except:
            gmm_fits = "Unimodal"

        # Volume distribution classification
        if num_peaks == 1:
            if dist_skewness < -0.5:
                volume_distribution = "Left-Skewed"
            elif dist_skewness > 0.5:
                volume_distribution = "Right-Skewed"
            else:
                volume_distribution = "Symmetric"
        elif num_peaks > 1:
            volume_distribution = "Multimodal"
        else:
            volume_distribution = "Flat or noisy"

        # Optional interpretation string
        distribution_insight = f"{num_peaks} peaks, {skew_label}, {kurtosis_label}"

        zones = {
            "VWAP": vwap,
            "VWAP - STD": vwap - std,
            "VWAP + STD": vwap + std,
            "VWAP - 2STD": vwap - 2 * std,
            "VWAP + 2STD": vwap + 2 * std,
            "MODE": mode_price
        }

        results.append({
            "properties": {
                "i": i,
                "window_days": window_days,
                "window_size": window_size,
                "window_step": window_step,
                "n_bins": n_bins,
            },
            "price_levels": price_levels,
            "volumes": volumes,
            "vwap": vwap,
            "std": std,
            "mode_price": mode_price,
            "zones": zones,
            "volume_distribution": volume_distribution,
            "skewness": dist_skewness,
            "kurtosis": dist_kurtosis,
            "num_peaks": num_peaks,
            "peaks_prices": peaks_prices,
            "peaks": peaks,
            "gmm_fit": gmm_fits,
            "skew_label": skew_label,
            "kurtosis_label": kurtosis_label,
            "distribution_insight": distribution_insight
        })
    return results


In [ ]:
def distribution_animation(results_dict: Dict[str, Any],
                           candles_df: pd.DataFrame):    # Extract series
    skews = [r["skewness"] for r in results_dict]
    kurts = [r["kurtosis"] for r in results_dict]
    peaks = [r["num_peaks"] for r in results_dict]

    # Define a color map for different distribution types
    color_map = {
        "Right-Skewed": "orange",
        "Left-Skewed": "red",
        "Symmetric": "green",
        "Multimodal (Bimodal/Trimodal)": "purple",
        "Flat or noisy": "gray"
    }

    # Helper to get color based on distribution
    def get_color(distribution):
        return color_map.get(distribution, "rgba(0, 150, 255, 0.6)")  # default blue

    # Create subplot layout
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.06,
        row_heights=[0.5, 0.15, 0.15, 0.15],
        # subplot_titles=("Volume by Price", "Skew", "Kurtosis", "Number of Peaks")
    )

    # Add initial volume-by-price (frame 0)
    first = results_dict[0]
    fig.add_trace(go.Bar(
        y=first["volumes"],
        x=first["price_levels"],
        # orientation="h",
        marker_color=get_color(first["volume_distribution"]),
        name="Volume by Price"
    ), row=1, col=1)

    # Add static metric trails (gray)
    fig.add_trace(go.Scatter(x=skews, y=["Skew"] * len(skews),
                             mode="lines+markers", line=dict(color="gray", dash="dot"),
                             marker=dict(size=3), showlegend=False), row=2, col=1)

    fig.add_trace(go.Scatter(x=kurts, y=["Kurtosis"] * len(kurts),
                             mode="lines+markers", line=dict(color="gray", dash="dot"),
                             marker=dict(size=3), showlegend=False), row=3, col=1)

    fig.add_trace(go.Scatter(x=peaks, y=["# Peaks"] * len(peaks),
                             mode="lines+markers", line=dict(color="gray", dash="dot"),
                             marker=dict(size=3), showlegend=False), row=4, col=1)

    # Create animation frames
    frames = []
    for i, r in enumerate(results_dict):
        frame = go.Frame(
            data=[
                go.Bar(
                    y=r["volumes"],
                    x=r["price_levels"],
                    # orientation="h",
                    marker_color=get_color(r["volume_distribution"])
                ),
                go.Scatter(x=[skews[i]], y=["Skew"], mode="markers",
                           marker=dict(size=10)),
                go.Scatter(x=[kurts[i]], y=["Kurtosis"], mode="markers",
                           marker=dict(size=10)),
                go.Scatter(x=[peaks[i]], y=["# Peaks"], mode="markers",
                           marker=dict(size=10, color="violet"))
            ],
            name=f"frame-{i}",
            layout=go.Layout(title_text=r["distribution_insight"])
        )
        frames.append(frame)

    fig.frames = frames

    # Axis config
    fig.update_layout(
        xaxis=dict(title="Price", range=[candles_df.close.min(), candles_df.close.max()]),
        xaxis2=dict(range=[-3.0, 3.0]),
        xaxis3=dict(range=[-5, 5]),
        xaxis4=dict( range=[0, 4])
    )

    # Animation controls
    fig.update_layout(
        height=800,
        updatemenus=[{
            "type": "buttons",
            "buttons": [{
                "label": "Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True}]
            }]
        }]
    )

    fig.show()

window_days = 14
window_size = 60 * 24 * window_days
window_step = 60 * 4
tick_size = 0.001
n_bins = 15

results = analyze_distribution(candles_df=candles.data)
df = pd.DataFrame(results)

In [ ]:
df.groupby(["volume_distribution", "skew_label", "kurtosis_label", "gmm_fit"])["vwap"].describe()


In [ ]:
skewness_filter: List[str] = None
kurtosis_filter: List[str] = None
num_peaks_filter: List[int] = [2]
results_filtered = [
        r for r in results if (
            (skewness_filter is None or r["skew_label"] in skewness_filter) and
            (kurtosis_filter is None or r["kurtosis_label"] in kurtosis_filter) and
            (num_peaks_filter is None or r["num_peaks"] in num_peaks_filter)
        )
    ]
distribution_animation(results_dict=results_filtered, candles_df=candles.data)


In [ ]:

# Tus datos agrupados
data = {
    "skew_label": [
        "Right-skewed (tail toward high prices)", "Right-skewed (tail toward high prices)",
        "Right-skewed (tail toward high prices)", "Symmetric", "Symmetric"
    ],
    "kurtosis_label": [
        "Leptokurtic (peaked, fat tails)", "Mesokurtic (normal shape)",
        "Platykurtic (flat, light tails)", "Mesokurtic (normal shape)",
        "Platykurtic (flat, light tails)"
    ],
    "gmm_fit": ["Bimodal"] * 5,
    "volume_distribution": ["Multimodal"] * 5,
    "mean": [5.7808, 5.7984, 5.7983, 5.7325, 5.7528],
    "std": [0.0818, 0.0543, 0.0326, 0.0034, 0.0222],
    "count": [147, 70, 71, 13, 54]
}

df = pd.DataFrame(data)
df["group"] = df["skew_label"] + "<br>" + df["kurtosis_label"]

# Expandir los valores para simular distribuciones
rows = []
for _, row in df.iterrows():
    samples = np.random.normal(loc=row["mean"], scale=row["std"], size=int(row["count"]))
    rows += [{"group": row["group"], "vwap": s} for s in samples]

expanded_df = pd.DataFrame(rows)

# Gráfico violin
fig = px.violin(expanded_df, y="vwap", x="group", box=True, points="all", title="VWAP Distribution by Skew and Kurtosis")
fig.update_layout(xaxis_title="Group (Skew + Kurtosis)", yaxis_title="VWAP")
fig.show()


In [ ]:
example = results[-1]
example

In [ ]:
import math

# i can get through 2 ways:
# 1) size orders according to current volume weights
# 2) truncate price levels to get only what distribution is giving me
peaks = example["peaks"]
prices = [round(math.ceil(price / tick_size) * tick_size, 3) for price in example["price_levels"]]

In [ ]:
sizes = example["volumes"] / example["volumes"].sum()

In [ ]:
print(peaks)
print(example["peaks_prices"])
print(list(zip(prices, sizes)))

In [ ]:
list(zip(prices, sizes))